# Lab 5F — SHAP Explainability

Per-prediction attribution for the deployed XGBoost model: **why** did the model predict this outcome for this client?

**Prerequisites:** Lab 3A has run, so a model version exists in the MLflow registry and the `training_data` Iceberg table is populated. Athena is the default data source; it falls back to a local CSV only if you flip `DATA_SOURCE = "csv"`.

**What you'll get:**
- Global feature importance — which features matter overall
- Per-prediction waterfall plots — why each decision
- Feature dependence plots — non-linear interactions
- SHAP artifacts logged back to MLflow (linked to the training run that produced the model)

> The interactive force plot in section 9.3 needs JavaScript. It renders in JupyterLab; static viewers such as the GitHub/GitLab renderer strip it.


## 1. Setup and Configuration

In [ ]:
# Load the shared environment written once by lab0-setup/setup.ipynb.
# No per-lab discovery or hardcoding — lab0-setup is the single source of truth.
import os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd()
for _cand in [_p, *_p.parents]:
    if (_cand / '.env').exists():
        load_dotenv(_cand / '.env', override=False)
        _ENV_PATH = _cand / '.env'
        break
else:
    raise RuntimeError('No .env found. Run lab0-setup/setup.ipynb first.')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'bank-marketing-prediction')
print(f'\u2713 Loaded shared environment from {_ENV_PATH}')
print(f'  PROJECT_NAME={PROJECT_NAME}')

# --- lab5-only: install this monitoring package so `import src` resolves. ---
# Only lab5d/lab5f need it, so it lives here rather than in the shared
# lab0-setup. (Temporary: pending the planned rename of src/ to a real package
# name, after which this becomes an ordinary dependency.)
import sys, subprocess
# lab5 notebooks run from lab5-monitoring/; find the dir that holds pyproject.toml.
_lab5 = next((p for p in [Path.cwd(), *Path.cwd().parents]
              if (p / 'pyproject.toml').exists() and (p / 'src').is_dir()), Path.cwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(_lab5), '--quiet'],
               check=False)
print(f'\u2713 lab5 monitoring package installed from {_lab5}')


In [ ]:
import sys
print(f"Python: {sys.executable}")

# Verify SHAP is installed
import shap
print(f"SHAP version: {shap.__version__}")

In [ ]:
import os, sys, importlib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML imports
import mlflow
import xgboost as xgb
import shap

# The shared repo-root .env was written by lab0-setup/setup.ipynb.
# lab5/ (this dir) is still put on sys.path so src/ imports work.
project_root = Path.cwd()
_env_dir = next((p for p in [project_root, *project_root.parents]
                 if (p / '.env').exists()), None)
if _env_dir is None:
    raise RuntimeError('No .env found. Run lab0-setup/setup.ipynb first.')

# Force lab5/ onto sys.path and clear stale src modules
_lab5_str = str(project_root)
if _lab5_str not in sys.path:
    sys.path.insert(0, _lab5_str)
for key in [k for k in list(sys.modules.keys()) if k == 'src' or k.startswith('src.')]:
    del sys.modules[key]

from dotenv import load_dotenv
load_dotenv(_env_dir / '.env', override=True)
print(f"✓ Loaded environment from: {_env_dir / '.env'}")

# Project imports
from src.config.config import (
    MLFLOW_TRACKING_URI,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_MODEL_NAME,
    MLFLOW_MONITORING_EXPERIMENT_NAME,
    AWS_DEFAULT_REGION,
)
from src.config import schema
from src.utils.shap_utils import (
    load_model_from_mlflow,
    prepare_background_data,
    compute_shap_values,
    validate_shap_values,
    create_shap_visualizations,
    save_shap_artifacts_to_mlflow
)
from src.utils.mlflow_utils import setup_mlflow_tracking

# Configure matplotlib
plt.style.use('default')
sns.set_palette("husl")

print("✓ All imports successful")

In [ ]:
# Initialize MLflow
setup_mlflow_tracking(MLFLOW_TRACKING_URI)

print("MLflow Configuration:")
print(f"  Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"  Training Experiment: {MLFLOW_EXPERIMENT_NAME}")
print(f"  Monitoring Experiment: {MLFLOW_MONITORING_EXPERIMENT_NAME}")
print(f"  Model Registry: {MLFLOW_MODEL_NAME}")
print(f"  Region: {AWS_DEFAULT_REGION}")

## 2. Load Model from MLflow

We'll load the latest registered model from MLflow. You can also specify a specific run ID.

In [ ]:
# Find the latest registered model version. We use search_model_versions
# (MLflow 3 API) instead of the deprecated get_latest_versions(stages=...)
# — MLflow 3 removed `stages` in favor of aliases. Other places in this
# project that resolve versions (train.py, test_endpoint.py,
# log_monitoring_to_mlflow.py) all use search_model_versions; we match.
client = mlflow.tracking.MlflowClient()

try:
    versions = client.search_model_versions(f"name='{MLFLOW_MODEL_NAME}'")
    if not versions:
        print(f"⚠ No registered versions for model: {MLFLOW_MODEL_NAME}")
        print("  Run 1_training_pipeline.ipynb first to train a model.")
        MODEL_RUN_ID = None
    else:
        latest = max(versions, key=lambda v: int(v.version))
        MODEL_RUN_ID = latest.run_id
        print(f"Latest registered model:")
        print(f"  Name:    {latest.name}")
        print(f"  Version: {latest.version}")
        print(f"  Run ID:  {MODEL_RUN_ID}")
        print(f"  Status:  {latest.status}")
        aliases = getattr(latest, 'aliases', []) or []
        if aliases:
            print(f"  Aliases: {', '.join(aliases)}")
except Exception as e:
    print(f"Error searching model registry: {e}")
    print("Alternative: set MODEL_RUN_ID manually in the next cell.")
    MODEL_RUN_ID = None

In [ ]:
# Optional: Override with specific run ID
#MODEL_RUN_ID = "bbb5436e9bcb47e6ad7f4824f5df0c21"

if MODEL_RUN_ID:
    print(f"Loading model from run: {MODEL_RUN_ID}")
    model, feature_names = load_model_from_mlflow(MODEL_RUN_ID)
    
    print(f"\n✓ Model loaded successfully")
    print(f"  Type: {type(model).__name__}")
    print(f"  Features: {len(feature_names)}")
    print(f"\nFeature Names:")
    for i, feat in enumerate(feature_names, 1):
        print(f"  {i:2d}. {feat}")
else:
    print("⚠ No model loaded. Please set MODEL_RUN_ID above.")

## 3. Load Training Data

**Data Source Choice**: This notebook defaults to loading from **Athena's `training_data` table** (not CSV).

**Why Athena is Recommended**:
- ✅ **Exact match**: `training_data` contains the POST-preprocessing data that the model was trained on
- ✅ **Already encoded**: Categorical columns like `customer_gender` are already label-encoded to integers
- ✅ **Accurate SHAP**: Using the exact training distribution ensures SHAP values accurately reflect model behavior
- ✅ **Always available**: No need to export/download CSV files manually

**CSV Option** (legacy):
- ⚠️ CSV path references an old file that's no longer auto-generated
- ⚠️ May require manual encoding to match preprocessing
- ⚠️ Only use if you have a specific reason to avoid Athena

**TL;DR**: Leave `DATA_SOURCE = "athena"` unless you have a compelling reason to change it.

In [ ]:
# Configuration
DATA_SOURCE = "athena"  # "athena" (RECOMMENDED — reads training_data Iceberg table with preprocessed data)
                        # "csv" (legacy — requires manual CSV export; not auto-generated)
NUM_SAMPLES = 5000

if DATA_SOURCE == "athena":
    from src.train_pipeline.athena.athena_client import AthenaClient
    from src.config.config import ATHENA_TRAINING_TABLE

    print(f"Loading {NUM_SAMPLES} samples from Athena ({ATHENA_TRAINING_TABLE})...")
    athena_client = AthenaClient()
    
    # Query training_data table directly
    # NOTE: training_data is the POST-preprocessing table, so categorical columns
    # (job, marital, education, ...) are already label-encoded, not raw strings.
    # That is exactly what the model was trained on, so no extra encoding here.
    df = athena_client.read_table(ATHENA_TRAINING_TABLE, limit=NUM_SAMPLES)
    print(f"✓ Loaded {len(df)} rows from Athena")
    print("  (Data is already preprocessed - every feature is numeric)")
    
elif DATA_SOURCE == "csv":
    from src.config.config import CSV_TRAINING_DATA

    if not CSV_TRAINING_DATA.exists():
        raise FileNotFoundError(
            f"CSV not found: {CSV_TRAINING_DATA}\n"
            f"The project no longer auto-generates this CSV (switched to kagglehub).\n"
            f"Use DATA_SOURCE='athena' (recommended) or manually export from Athena."
        )
    print(f"Loading from CSV: {CSV_TRAINING_DATA}")
    df = pd.read_csv(CSV_TRAINING_DATA)
    if len(df) > NUM_SAMPLES:
        df = df.sample(n=NUM_SAMPLES, random_state=42)
    print(f"✓ Loaded {len(df)} rows from CSV")
    print("  ⚠️ CSV may not match current preprocessing - use 'athena' for accuracy")
else:
    raise ValueError(f"DATA_SOURCE must be 'athena' or 'csv', got: {DATA_SOURCE!r}")

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

TARGET_COLUMN = schema.target_column()
if TARGET_COLUMN in df.columns:
    subscription_rate = df[TARGET_COLUMN].mean()
    print(f"\nSubscription rate: {subscription_rate:.2%}")
    print(f"  Subscribed:     {int(df[TARGET_COLUMN].sum()):,}")
    print(f"  Not subscribed: {int((~df[TARGET_COLUMN].astype(bool)).sum()):,}")
else:
    print(f"⚠ Target column '{TARGET_COLUMN}' not found in data")

In [ ]:
# Prepare features and target
#
# The booster expects its features in the exact order it was trained on, so X is
# built from `feature_names` (the model's own list) rather than from the table's
# column order. Anything the model lists but the schema does not is not a real
# predictive feature, so it is zeroed rather than fed noise — if that count is
# not 0 below, the model and the schema have diverged.

print("Preparing feature data...")

# Identify which of the model's feature_names are real schema features
_schema_features = set(schema.feature_names())
_metadata_cols = [f for f in feature_names if f not in _schema_features]
_real_features = [f for f in feature_names if f in _schema_features]

if _metadata_cols:
    print(f"  ℹ Model includes {len(_metadata_cols)} metadata columns: {_metadata_cols}")
    print(f"  ℹ These will be zeroed out for SHAP (not real predictive features)")

# Build X with ALL model features (XGBoost needs exact column count/order)
X = pd.DataFrame(index=df.index)
for col in feature_names:
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
        X[col] = df[col].values
    elif col in df.columns:
        # Coerce non-numeric to numeric (e.g. client_id string -> NaN -> 0)
        X[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).values
    else:
        # Column not in data at all — fill with 0
        X[col] = 0

X = X.fillna(0).astype(np.float64)

y = df[TARGET_COLUMN].copy() if TARGET_COLUMN in df.columns else None

print(f"\nFeatures shape: {X.shape} (all {len(feature_names)} model features)")
print(f"  Real features:    {len(_real_features)}")
print(f"  Metadata (zeroed): {len(_metadata_cols)}")
print(f"Target shape:   {y.shape if y is not None else 'N/A'}")
print("✓ All features are numeric")

## 4. Prepare Background Dataset

SHAP requires a background dataset to estimate the expected value (baseline prediction). We use stratified sampling to preserve the subscribed / not-subscribed ratio.

In [ ]:
# Configuration
BACKGROUND_SAMPLES = 500
EXPLAIN_SAMPLES = 200  # Number of predictions to explain

# Prepare background data (stratified sampling)
print(f"Preparing background dataset ({BACKGROUND_SAMPLES} samples)...")
X_background = prepare_background_data(
    X, y,
    n_samples=BACKGROUND_SAMPLES,
    stratify=True if y is not None else False,
    random_state=42
)

# Prepare data to explain
print(f"\nPreparing explanation dataset ({EXPLAIN_SAMPLES} samples)...")
if len(X) > EXPLAIN_SAMPLES:
    # Sample diverse predictions (stratified)
    if y is not None:
        from sklearn.model_selection import train_test_split
        _, X_explain, _, y_explain = train_test_split(
            X, y, test_size=EXPLAIN_SAMPLES, stratify=y, random_state=42
        )
    else:
        X_explain = X.sample(n=EXPLAIN_SAMPLES, random_state=42)
        y_explain = None
else:
    X_explain = X
    y_explain = y

print(f"✓ Background data: {X_background.shape}")
print(f"✓ Explanation data: {X_explain.shape}")

if y_explain is not None:
    print(f"\nExplanation Set Subscription Rate: {y_explain.mean():.2%}")

## 5. Compute SHAP Values

We use TreeExplainer, which is optimized for XGBoost and provides exact Shapley values.

In [ ]:
import time

print("Computing SHAP values...\n")
print("This may take a few minutes depending on:")
print(f"  - Background samples: {len(X_background)}")
print(f"  - Explanation samples: {len(X_explain)}")
print(f"  - Features: {len(feature_names)}")
print("\nProgress: Starting...")

start_time = time.time()

# Compute SHAP values
explainer, shap_values = compute_shap_values(
    model,
    X_background,
    X_explain,
    check_additivity=False  # Set True for debugging, but slower
)

elapsed_time = time.time() - start_time

print(f"\n✓ SHAP computation complete!")
print(f"  Time: {elapsed_time:.2f} seconds")
print(f"  Rate: {len(X_explain) / elapsed_time:.1f} samples/second")
print(f"  SHAP values shape: {shap_values.shape}")
print(f"  Expected value (base): {explainer.expected_value}")

## 6. Validate SHAP Values

Verify the mathematical consistency: `base_value + sum(shap_values) ≈ model_prediction`

In [ ]:
# Get model predictions

# Ensure X_explain is numeric for XGBoost
X_explain_numeric = X_explain.copy()
for col in X_explain_numeric.columns:
    if X_explain_numeric[col].dtype == 'object' or X_explain_numeric[col].dtype.name == 'category':
        X_explain_numeric[col] = pd.to_numeric(X_explain_numeric[col], errors='coerce')
X_explain_numeric = X_explain_numeric.astype(np.float64)

if isinstance(model, xgb.Booster):
    dmatrix = xgb.DMatrix(X_explain_numeric)
    predictions = model.predict(dmatrix)
else:
    predictions = model.predict_proba(X_explain_numeric)[:, 1]  # P(subscribed)

print(f"Model Predictions:")
print(f"  Min: {predictions.min():.4f}")
print(f"  Max: {predictions.max():.4f}")
print(f"  Mean: {predictions.mean():.4f}")
print(f"  Median: {np.median(predictions):.4f}")

# Validate SHAP values
print("\nValidating SHAP values...")
validation_results = validate_shap_values(
    explainer,
    shap_values,
    predictions,
    tolerance=1e-4
)

if validation_results['passed']:
    print(f"\n✓ SHAP Validation PASSED")
    print(f"  Max error: {validation_results['max_error']:.2e}")
    print(f"  Mean error: {validation_results['mean_error']:.2e}")
else:
    print(f"\n✗ SHAP Validation FAILED")
    print(f"  Failed samples: {len(validation_results['failed_samples'])}")
    print(f"  Max error: {validation_results['max_error']:.2e}")
    print(f"\nFirst few failures:")
    for sample in validation_results['failed_samples'][:3]:
        print(f"  Sample {sample['index']}: error = {sample['error']:.2e}")

## 7. Global Interpretability: Feature Importance

Which features are most important overall?

In [ ]:
# Calculate mean absolute SHAP values
mean_abs_shap = np.abs(shap_values).mean(axis=0)

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': mean_abs_shap
}).sort_values('importance', ascending=False)

print("Top 10 Features by SHAP Importance:\n")
print(feature_importance_df.head(10).to_string(index=False))

# Visualize top 15 features
plt.figure(figsize=(10, 8))
top_features = feature_importance_df.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Mean |SHAP Value|', fontsize=12)
plt.title('Top 15 Features by SHAP Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\n💡 Insight: The top feature '{top_features.iloc[0]['feature']}' has {top_features.iloc[0]['importance']:.4f} mean absolute SHAP value.")

## 8. SHAP Summary Plots

**Bar plot**: Shows feature importance (mean absolute SHAP)

**Beeswarm plot**: Shows feature value distribution and impact on predictions

In [ ]:
# Summary plot (bar) - Feature importance
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_explain, plot_type="bar", max_display=20, show=False)
plt.title("SHAP Feature Importance", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation: Features are ranked by their average impact on model output magnitude.")

In [ ]:
# Summary plot (beeswarm) - Feature value impact
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_explain, max_display=20, show=False)
plt.title("SHAP Summary Plot (Feature Impact by Value)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("  - Each dot is a sample")
print("  - X-axis: SHAP value (left = lowers P(subscribed), right = raises it)")
print("  - Color: Feature value (pink = high, blue = low)")
print("  - Vertical spread: Number of samples with that SHAP value")

## 9. Local Interpretability: Individual Predictions

### 9.1 Waterfall Plot - Highest Positive-Prediction Case

In [ ]:
# The client the model is most confident will subscribe.
top_idx = np.argmax(predictions)
top_prob = predictions[top_idx]

print(f"Analyzing highest positive-prediction case:")
print(f"  Sample index: {top_idx}")
print(f"  Positive-class probability: {top_prob:.4f}")
if y_explain is not None:
    actual_label = "Subscribed" if y_explain.iloc[top_idx] else "Not subscribed"
    print(f"  Actual label: {actual_label}")

# Waterfall plot
plt.figure(figsize=(10, 8))
base_val = explainer.expected_value if not isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value[1]
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[top_idx],
        base_values=base_val,
        data=X_explain.iloc[top_idx],
        feature_names=list(X_explain.columns)
    ),
    max_display=15,
    show=False
)
plt.title(f"SHAP Waterfall Plot - Highest Positive-Prediction Case (prob={top_prob:.4f})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("  - Base value (E[f(x)]): Expected P(subscribed) across the background set")
print("  - Red bars: Features pushing the prediction HIGHER (toward subscribing)")
print("  - Blue bars: Features pushing it LOWER (toward not subscribing)")
print("  - Final prediction: Base value + sum of all feature contributions")

### 9.2 Waterfall Plot - Lowest Negative-Prediction Case

In [ ]:
# The client the model is most confident will NOT subscribe.
low_idx = np.argmin(predictions)
low_prob = predictions[low_idx]

print(f"Analyzing lowest negative-prediction case:")
print(f"  Sample index: {low_idx}")
print(f"  Positive-class probability: {low_prob:.4f}")
if y_explain is not None:
    actual_label = "Subscribed" if y_explain.iloc[low_idx] else "Not subscribed"
    print(f"  Actual label: {actual_label}")

# Waterfall plot
plt.figure(figsize=(10, 8))
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[low_idx],
        base_values=base_val,
        data=X_explain.iloc[low_idx],
        feature_names=list(X_explain.columns)
    ),
    max_display=15,
    show=False
)
plt.title(f"SHAP Waterfall Plot - Lowest Negative-Prediction Case (prob={low_prob:.4f})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Insight: Compare this to the likely-subscriber above to see how the feature contributions differ.")

### 9.3 Force Plot (Interactive)

In [ ]:
# Force plot for a single prediction (interactive HTML)
sample_idx = top_idx  # the likely-subscriber identified above

print(f"Creating force plot for sample {sample_idx}...")
shap.initjs()
force_plot = shap.force_plot(
    base_val,
    shap_values[sample_idx],
    X_explain.iloc[sample_idx],
    matplotlib=False
)

display(force_plot)

print("\n📊 Interactive force plot displayed above.")
print("  - Hover over features to see their names and values")
print("  - Red: Features increasing P(subscribed)")
print("  - Blue: Features decreasing P(subscribed)")

### 9.4 Decision Plot (Compare Multiple Predictions)

In [ ]:
# Decision plot - compare multiple predictions
sample_size = min(50, len(X_explain))
sample_indices = np.random.choice(len(X_explain), size=sample_size, replace=False)

plt.figure(figsize=(12, 8))
shap.decision_plot(
    base_val,
    shap_values[sample_indices],
    X_explain.iloc[sample_indices],
    show=False
)
plt.title(f"SHAP Decision Plot ({sample_size} Predictions)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("  - Each line represents one prediction")
print("  - Y-axis: Features (ordered by importance)")
print("  - X-axis: Model output (P(subscribed))")
print("  - Lines moving right: Features increasing P(subscribed)")
print("  - Lines moving left: Features decreasing P(subscribed)")

## 10. Feature Interactions: Dependence Plots

Dependence plots show how a feature's value affects its SHAP value, revealing interactions with other features.

In [ ]:
# Get top 3 features
top_3_features = feature_importance_df.head(3)['feature'].tolist()

print(f"Creating dependence plots for top 3 features:")
for rank, feature in enumerate(top_3_features, 1):
    print(f"  {rank}. {feature}")

# Create dependence plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feature in enumerate(top_3_features):
    feat_idx = feature_names.index(feature)
    
    plt.sca(axes[idx])
    shap.dependence_plot(
        feat_idx,
        shap_values,
        X_explain,
        show=False
    )
    axes[idx].set_title(f"Dependence: {feature}", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("  - X-axis: Feature value")
print("  - Y-axis: SHAP value (impact on prediction)")
print("  - Color: Another feature (automatic interaction detection)")
print("  - Vertical scatter at same X: Interactions with other features")

## 11. Save Artifacts

Save all plots and log to MLflow.

In [ ]:
# Create output directory
output_dir = Path("shap_output")
output_dir.mkdir(exist_ok=True)

print(f"Saving SHAP artifacts to: {output_dir}")

# Generate all visualizations
plot_paths = create_shap_visualizations(
    explainer,
    shap_values,
    X_explain,
    output_dir,
    max_display=20
)

print(f"\n✓ Saved {len(plot_paths)} visualizations:")
for plot_name, plot_path in plot_paths.items():
    print(f"  - {plot_path}")

In [ ]:
# Save SHAP values as CSV
shap_df = pd.DataFrame(shap_values, columns=feature_names)
shap_csv_path = output_dir / "shap_values.csv"
shap_df.to_csv(shap_csv_path, index=False)
print(f"✓ Saved SHAP values: {shap_csv_path}")

# Save feature importance
feature_importance_df.to_csv(output_dir / "feature_importance.csv", index=False)
print(f"✓ Saved feature importance: {output_dir / 'feature_importance.csv'}")

## 12. Log to MLflow

Log all SHAP artifacts to MLflow for tracking and comparison.

In [ ]:
print("Logging SHAP artifacts to MLflow...")

shap_run_id = save_shap_artifacts_to_mlflow(
    training_run_id=MODEL_RUN_ID,
    shap_values=shap_values,
    plot_paths=plot_paths,
    feature_names=feature_names,
    experiment_name=MLFLOW_MONITORING_EXPERIMENT_NAME
)

print(f"\n✓ SHAP artifacts logged to MLflow")
print(f"  Run ID: {shap_run_id}")
print(f"  Experiment: {MLFLOW_MONITORING_EXPERIMENT_NAME}")
print(f"  Training Run: {MODEL_RUN_ID}")

# Construct MLflow UI URL
mlflow_browser_url = os.getenv('MLFLOW_TRACKING_BROWSER_URL', '')
if mlflow_browser_url:
    print(f"\n🔗 View in MLflow UI:")
    print(f"  {mlflow_browser_url}")

## 13. Summary and Key Insights

In [ ]:
print("="*80)
print("SHAP EXPLAINABILITY ANALYSIS SUMMARY")
print("="*80)

print(f"\n📊 Dataset Statistics:")
print(f"  Total samples analyzed: {len(X_explain)}")
print(f"  Features: {len(feature_names)}")
print(f"  Background samples: {len(X_background)}")

print(f"\n🎯 Model Performance:")
print(f"  Average P(subscribed): {predictions.mean():.4f}")
print(f"  Min P(subscribed): {predictions.min():.4f}")
print(f"  Max P(subscribed): {predictions.max():.4f}")

print(f"\n🔍 SHAP Analysis Results:")
print(f"  Validation: {validation_results['passed']}")
print(f"  Max error: {validation_results['max_error']:.2e}")
print(f"  Computation time: {elapsed_time:.2f} seconds")

print(f"\n⭐ Top 5 Most Important Features:")
# enumerate, not iterrows: the DataFrame index here is the original feature
# position, so `rank+1` from iterrows would print positions, not ranks 1-5.
for rank, (_, row) in enumerate(feature_importance_df.head(5).iterrows(), 1):
    print(f"  {rank}. {row['feature']:30s} (SHAP: {row['importance']:.4f})")

print(f"\n💾 Artifacts Saved:")
print(f"  Visualizations: {len(plot_paths)} plots")
print(f"  SHAP values CSV: {shap_csv_path}")
print(f"  MLflow run: {shap_run_id}")

print("\n" + "="*80)
print("✓ SHAP ANALYSIS COMPLETE")
print("="*80)

print("\n💡 Next Steps:")
print("  1. Review feature importance to validate model behavior")
print("  2. Analyze waterfall plots to understand specific predictions")
print("  3. Examine dependence plots for feature interactions")
print("  4. Compare SHAP results across model versions in MLflow")
print("  5. Use insights to improve feature engineering or model architecture")